In [9]:
# Import libraries
import os
import re
import numpy as np
import pandas as pd

import nltk

nltk.download("stopwords", quiet=True)
nltk.download("punkt_tab", quiet=True)
nltk.download("punkt", quiet=True)
nltk.download("averaged_perceptron_tagger", quiet=True)
nltk.download("wordnet", quiet=True)

from nltk.corpus import stopwords
from nltk.tokenize import TreebankWordTokenizer, word_tokenize
from nltk.stem import PorterStemmer, SnowballStemmer
from nltk.stem import WordNetLemmatizer

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

import matplotlib.pyplot as plt
import seaborn as sns

In [10]:
data = pd.read_csv(
    "C:/Users/faiz/Desktop/Projectss/nlp-engineering/data/ai_ml_course_reviews.csv"
)

In [11]:
data.head(2)

,review_text,sentiment
0,The instructor explained backpropagation in a ...,1
1,Lab sessions were well-structured and the hand...,1


In [12]:
# ============================================================================
# COMPLETE PREPROCESSING PIPELINE
# ============================================================================
# Using Treebank tokenizer + Porter Stemmer + keep "not" stopwords
tokenizer = TreebankWordTokenizer()
stemmer = PorterStemmer()

# Stopwords but keep negation words for sentiment
stop_words_with_not = set(stopwords.words("english")) - {"not", "no", "n't", "never"}


def preprocess_text(text):
    # 1. Clean text (lowercase + noise removal)
    text = clean_text(text)
    # 2. Tokenize
    tokens = tokenizer.tokenize(text)
    # 3. Remove stopwords (keep "not")
    tokens = [t for t in tokens if t not in stop_words_with_not]
    # 4. Stem
    tokens = [stemmer.stem(t) for t in tokens]
    return " ".join(tokens)


# Apply preprocessing
data["processed_text"] = data["review_text"].apply(preprocess_text)
print("Complete Preprocessing Pipeline Applied!")
print(data[["review_text", "processed_text"]].head(3))

NameError: name 'clean_text' is not defined

In [ ]:
# ============================================================================
# STEP 2: Tokenization - Whitespace vs Linguistic
# ============================================================================
sample_text = "The course didn't cover advanced topics, but the labs were great!"

# Whitespace tokenization (crude)
whitespace_tokens = sample_text.lower().split()
print("Sample text:", sample_text)
print("\n--- Whitespace Tokenization:", whitespace_tokens)

# Linguistic tokenization (NLTK word_tokenize)
nltk_tokens = word_tokenize(sample_text.lower())
print("--- NLTK word_tokenize:", nltk_tokens)

# Treebank tokenizer (more refined)
treebank_tokens = TreebankWordTokenizer().tokenize(sample_text.lower())
print("--- TreebankWordTokenizer:", treebank_tokens)

print("\n>>> Why it matters:")
print("- Whitespace: fast but splits 'didn't' → ['didn't'] (loses 'did' and 'not')")
print(
    "- NLTK/Treebank: splits 'didn't' → ['did', 'n't'] (preserves 'not' for sentiment!)"
)

Sample text: The course didn't cover advanced topics, but the labs were great!

--- Whitespace Tokenization: ['the', 'course', "didn't", 'cover', 'advanced', 'topics,', 'but', 'the', 'labs', 'were', 'great!']
--- NLTK word_tokenize: ['the', 'course', 'did', "n't", 'cover', 'advanced', 'topics', ',', 'but', 'the', 'labs', 'were', 'great', '!']
--- TreebankWordTokenizer: ['the', 'course', 'did', "n't", 'cover', 'advanced', 'topics', ',', 'but', 'the', 'labs', 'were', 'great', '!']

>>> Why it matters:
- Whitespace: fast but splits 'didn't' → ['didn't'] (loses 'did' and 'not')
- NLTK/Treebank: splits 'didn't' → ['did', 'n't'] (preserves 'not' for sentiment!)


In [ ]:
# ============================================================================
# STEP 1: Lowercasing + Noise Removal
# ============================================================================
def clean_text(text):
    """Comprehensive text cleaning with regex patterns."""
    # Lowercase
    text = text.lower()
    # Remove HTML tags
    text = re.sub(r"<[^>]+>", "", text)
    # Remove URLs
    text = re.sub(r"http\S+|www\.\S+", "", text)
    # Remove email addresses
    text = re.sub(r"\S+@\S+", "", text)
    # Remove special characters and punctuation
    text = re.sub(r"[^a-zA-Z\s]", "", text)
    # Remove extra whitespace
    text = re.sub(r"\s+", " ", text).strip()
    return text


# Apply cleaning
data["cleaned_text"] = data["review_text"].apply(clean_text)
print("Step 1: Lowercasing + Noise Removal")
print(data[["review_text", "cleaned_text"]].head(2))

NameError: name 'data' is not defined

In [ ]:
# ============================================================================
# STEP 3: Stopword Removal
# ============================================================================
stop_words = set(stopwords.words("english"))


# Standard stopword removal
def remove_stopwords(text):
    tokens = word_tokenize(text)
    return " ".join([w for w in tokens if w not in stop_words])


# With "not" preserved (important for sentiment!)
stop_words_with_not = set(stopwords.words("english")) - {"not", "no", "n't", "never"}


def remove_stopwords_keep_not(text):
    tokens = word_tokenize(text)
    return " ".join([w for w in tokens if w not in stop_words_with_not])


sample = "I did not like the course, it was not helpful"
print("Original:", sample)
print("Standard stopwords removed:", remove_stopwords(sample))
print("With 'not' preserved:", remove_stopwords_keep_not(sample))

print("\n>>> When to keep stopwords:")
print("- 'not', 'no', 'never', 'n't' carry sentiment (negative/positive)")
print("- Removing them loses negation: 'not good' → 'good' (wrong!)")

Original: I did not like the course, it was not helpful
Standard stopwords removed: I like course , helpful
With 'not' preserved: I not like course , not helpful

>>> When to keep stopwords:
- 'not', 'no', 'never', 'n't' carry sentiment (negative/positive)
- Removing them loses negation: 'not good' → 'good' (wrong!)


In [ ]:
# ============================================================================
# STEP 4: Stemming vs Lemmatization Comparison
# ============================================================================
porter = PorterStemmer()
snowball = SnowballStemmer("english")
lemmatizer = WordNetLemmatizer()

words = ["running", "ran", "runs", "better", "studies", "students"]

print("Word     | Porter | Snowball | Lemmatizer (noun)")
print("-" * 55)
for word in words:
    p = porter.stem(word)
    s = snowball.stem(word)
    l = lemmatizer.lemmatize(word)  # Default: noun
    print(f"{word:10}| {p:8}| {s:8}| {l}")

# Lemmatizer with POS tagging
print("\n>>> With POS tagging (verb):")
for word in words:
    l_verb = lemmatizer.lemmatize(word, pos="v")
    print(f"{word} → {l_verb}")

print("\n>>> Tradeoff:")
print("- Stemming: faster, cruder (running→run, but studies→studi)")
print("- Lemmatization: slower, preserves meaning (studies→study)")
print("- Lemmatization needs POS tagging for accuracy")

Word     | Porter | Snowball | Lemmatizer (noun)
-------------------------------------------------------
running   | run     | run     | running
ran       | ran     | ran     | ran
runs      | run     | run     | run
better    | better  | better  | better
studies   | studi   | studi   | study
students  | student | student | student

>>> With POS tagging (verb):
running → run
ran → run
runs → run
better → better
studies → study
students → students

>>> Tradeoff:
- Stemming: faster, cruder (running→run, but studies→studi)
- Lemmatization: slower, preserves meaning (studies→study)
- Lemmatization needs POS tagging for accuracy


In [13]:
# ============================================================================
# STEP 5: TF-IDF Vectorization with N-grams Configuration
# ============================================================================
# Split data
X = data["processed_text"]
y = data["sentiment"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# TF-IDF with unigram + bigram, max_features, min_df, max_df
tfidf = TfidfVectorizer(
    ngram_range=(1, 2),  # unigrams + bigrams
    max_features=1000,  # limit features
    min_df=2,  # ignore terms in < 2 docs
    max_df=0.95,  # ignore terms in > 95% docs
    sublinear_tf=True,  # apply log scaling
)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

print(f"Training samples: {len(X_train)}")
print(f"Testing samples: {len(X_test)}")
print(f"TF-IDF features: {X_train_tfidf.shape[1]}")
print(f"\nConfig: ngram_range=(1,2), max_features=1000, min_df=2, max_df=0.95")

# Train model
model = LogisticRegression(max_iter=1000)
model.fit(X_train_tfidf, y_train)

# Evaluate
y_pred = model.predict(X_test_tfidf)
accuracy = accuracy_score(y_test, y_pred)
print(f"\nModel Accuracy: {accuracy:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

KeyError: 'processed_text'